# Assignment 4
**Name:** Waris  
**Roll No:** 1024170155

In [1]:
import pandas as pd

## Q1: Build Your Personalized Knowledge Base

Take your college roll number and extract its digits. Build a pandas DataFrame with exactly **6 FAQ entries**:

- **4 fixed entries** (given in the table below)
- **2 entries constructed from your own roll number digits**, as follows:

### Steps to construct the 2 custom entries
1. Take the **last two digits** of your roll number.
2. For each digit `d`, compute:
```python
   category = ["billing", "account", "general"][d % 3]
```
3. Invent one realistic **question + answer + 3 keywords** per entry that fits the assigned category.
   - *Example:* if `d % 3` gives `"account"`, write a question like *"How do I update my registered mobile number?"*

### Calculation for Roll Number: 1024170155
- Last two digits: `5`, `5`
- Digit 1 (`5`): `5 % 3 = 2` -> `category = ["billing", "account", "general"][2]` = **"general"**
- Digit 2 (`5`): `5 % 3 = 2` -> `category = ["billing", "account", "general"][2]` = **"general"**

In [2]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or netbanking.", 
     "keywords": "pay payment upi fee", "category": "billing"}
]

# Roll Number: 1024170155 (Waris)
# Last two digits: 5, 5 -> 5 % 3 = 2 -> "general", 5 % 3 = 2 -> "general"
custom_entries = [
    {"question": "where is your main office located",
     "answer": "Our main office is located in Sector 17, Chandigarh.",
     "keywords": "office location address where", "category": "general"},
    {"question": "how can i contact customer support",
     "answer": "You can reach customer support via email at support@example.com or call 1800-123-456.",
     "keywords": "contact support helpline call", "category": "general"}
]

faq_entries = fixed_entries + custom_entries
df = pd.DataFrame(faq_entries)
print(df)

                             question  ... category
0              what is the annual fee  ...  billing
1               how to reset password  ...  account
2         what are your working hours  ...  general
3               how can i pay the fee  ...  billing
4   where is your main office located  ...  general
5  how can i contact customer support  ...  general

[6 rows x 4 columns]


## Q2 : Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence

In [3]:
def score_query(query, df):
    query_words = set(query.lower().split())
    if not query_words:
        return pd.DataFrame(columns=["question", "answer", "category", "confidence"])
    results = []

    for _, row in df.iterrows():
        entry_words = set(row["question"].lower().split()) | set(row["keywords"].lower().split())
        matches = query_words & entry_words
        score = len(matches)

        if score > 0:
            confidence = score / len(query_words)
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": round(confidence, 2)
            })

    results.sort(key=lambda x: x["confidence"], reverse=True)
    return pd.DataFrame(results)

score_query("how do I pay my fee", df)

,question,answer,category,confidence
0,how can i pay the fee,"You can pay via UPI, card, or netbanking.",billing,0.67
1,how can i contact customer support,You can reach customer support via email at su...,general,0.33
2,what is the annual fee,The annual fee is Rs 500.,billing,0.17
3,how to reset password,Go to Settings > Reset Password.,account,0.17


## Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [4]:
def same_category(category_name, df):
    matches = df[df["category"] == category_name]
    return matches["question"]

# Personalized entries for roll number 1024170155 belong to "general"
print(same_category("general", df))

2           what are your working hours
4     where is your main office located
5    how can i contact customer support
Name: question, dtype: str


## Q4 : Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [5]:
roll_no = "1024170155"
entry_index = 0

new_keyword = input("Enter a new keyword to add : ").strip()
df.at[entry_index, "keywords"] = f"{df.at[entry_index, 'keywords']} {new_keyword}".strip()
csv_filename = f"{roll_no}_faq_data.csv"
df.to_csv(csv_filename, index=False)
print(f"Updated entry {entry_index} keywords: {df.at[entry_index, 'keywords']}")
print(f"Saved to {csv_filename}")

Enter a new keyword to add : tuition
Updated entry 0 keywords: fee cost price charge tuition
Saved to 1024170155_faq_data.csv


## Q5 : Using groupby, print how many FAQ entries you have per category.

In [6]:
print(df.groupby("category").size())

category
account    1
billing    2
general    3
dtype: int64


## Q6 : Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [7]:
def score_query(query, df):
    query_words = set(query.lower().split())
    if not query_words:
        print("No match found.")
        return
    results = []
    for _, row in df.iterrows():
        entry_words = set(row["question"].lower().split()) | set(row["keywords"].lower().split())
        score = len(query_words & entry_words)
        if score > 0:
            confidence = score / len(query_words)
            results.append({
                "question": row["question"],
                "confidence": round(confidence, 2)
            })
    if not results:
        print("No match found.")
        return
    # find the highest confidence
    best_score = max(r["confidence"] for r in results)
    # keep only entries with that highest score
    best_matches = [r for r in results if r["confidence"] == best_score]

    if len(best_matches) > 1:
        print("Tie! Multiple entries match equally well:")
    else:
        print("Best match:")

    for r in best_matches:
        print(r)


# Query that ties (matches both "fee" entries)
print("Query: 'fee'")
score_query("fee", df)
print()
# Query that doesn't tie
print("Query: 'how do I pay my fee'")
score_query("how do I pay my fee", df)

Query: 'fee'
Tie! Multiple entries match equally well:
{'question': 'what is the annual fee', 'confidence': 1.0}
{'question': 'how can i pay the fee', 'confidence': 1.0}

Query: 'how do I pay my fee'
Best match:
{'question': 'how can i pay the fee', 'confidence': 0.67}
